# Notebook 1 — Phase A: Data Building

Produces `sft_data_L1.jsonl`, `sft_data_L2.jsonl`, `dpo_pairs.jsonl` on Google Drive.

**Resumable:** skip-if-exists logic on every output file — safe to re-run after session timeout.

In [ ]:
# Cell 1 — Mount Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate tqdm datasets math-verify sympy peft

import os
BASE_DIR = "/content/drive/MyDrive/P-ALIGN"
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# Cell 2 — Download Model (skip if already on Drive)
from huggingface_hub import snapshot_download

MODEL_DIR = f"{BASE_DIR}/models/Qwen2.5-1.5B-Instruct"
if not os.path.exists(f"{MODEL_DIR}/config.json"):
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL_DIR)
    print(f"Downloaded → {MODEL_DIR}")
else:
    print(f"[Resume] Model already at {MODEL_DIR}")

In [ ]:
# Cell 3 — Load & Join Datasets → prefixes.jsonl
import re, json
from datasets import load_dataset

PREFIX_FILE = f"{BASE_DIR}/data/prefixes.jsonl"

if os.path.exists(PREFIX_FILE):
    with open(PREFIX_FILE) as f:
        rows = [json.loads(l) for l in f]
    print(f"[Resume] Loaded {len(rows)} rows from {PREFIX_FILE}")
else:
    # ── Step A: parse P-ALIGN (Alpaca format) ──────────────────────────────
    palign = load_dataset("qizheyanger/P-ALIGN", split="train")
    print(f"P-ALIGN columns: {palign.column_names}")

    _Q_RE = re.compile(
        r"\*Question\*\s*[:\-]?\s*(.*?)\s*\*Prefix\*\s*[:\-]?\s*(.*)",
        re.DOTALL | re.IGNORECASE,
    )

    def parse_instruction(instruction: str):
        m = _Q_RE.search(instruction)
        if m:
            return m.group(1).strip(), m.group(2).strip()
        return instruction.strip(), ""

    palign_parsed = []
    for item in palign:
        q, p = parse_instruction(item["instruction"])
        palign_parsed.append({"question": q, "prefix": p})

    # ── Step B: load s1K-1.1 ───────────────────────────────────────────────
    s1k = load_dataset("simplescaling/s1K-1.1", split="train")
    print(f"s1K-1.1 columns: {s1k.column_names}")

    s1k_lookup = {}
    for item in s1k:
        key = item["question"].strip()
        s1k_lookup[key] = {
            "full_cot": item["deepseek_thinking_trajectory"],
            "answer":   str(item["solution"]).strip(),
        }
    print(f"s1K-1.1 lookup: {len(s1k_lookup)} entries")

    # ── Step C: join on question text ─────────────────────────────────────
    rows, n_miss = [], 0
    for i, parsed in enumerate(palign_parsed):
        q = parsed["question"]
        lookup = s1k_lookup.get(q)
        if lookup is None:
            q_norm = re.sub(r"\s+", " ", q.lower().strip())
            lookup = next(
                (v for k, v in s1k_lookup.items()
                 if re.sub(r"\s+", " ", k.lower().strip()) == q_norm),
                None,
            )
        if lookup is None:
            n_miss += 1
            continue
        rows.append({
            "id":       i,
            "question": q,
            "prefix":   parsed["prefix"],
            "answer":   lookup["answer"],
            "full_cot": lookup["full_cot"],
        })

    print(f"Joined {len(rows)} samples ({n_miss} unmatched, dropped)")
    if n_miss / max(len(palign_parsed), 1) > 0.05:
        print("[warn] >5% unmatched — consider adding unicodedata.normalize('NFC') to join keys.")

    with open(PREFIX_FILE, "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved → {PREFIX_FILE}")

In [ ]:
# Cell 4 — Load Student Model for Generation
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
# Cell 5 — Hyperparameters
K          = 4      # continuations per question
TEMP       = 0.8
TOP_P      = 0.95
MAX_NEW    = 2048
TAU_LOW    = 0.2    # prefix-feedback trigger
T_MAX      = 3      # max feedback attempts
LAMBDA_LEN = 0.0    # length penalty (0 = disabled)
EPSILON    = 1e-6
CLIP_C     = 3.0

In [ ]:
# Cell 6 — Prompt Builder
def build_prompt(question: str, prefix: str) -> str:
    return (
        "Please continue from the draft and solve the problem step by step, "
        "and put your final answer within \\boxed{}. "
        "I will provide you with some prior knowledge as a draft to assist you.\n"
        f"*Question*: {question}\n"
        f"*Prefix*: {prefix}"
    )

In [ ]:
# Cell 7 — K-Continuation Sampler (Step 3)
from collections import Counter

def sample_k_continuations(question: str, prefix: str, k: int = K) -> list:
    prompt = build_prompt(question, prefix)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    continuations = []
    for _ in range(k):
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW,
                do_sample=True,
                temperature=TEMP,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        cont = tokenizer.decode(
            out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        continuations.append(cont)

    answers = [extract_boxed(c) for c in continuations]
    if len(Counter(answers)) == 1:
        print("  [warn] All K continuations identical — consider raising temperature.")
    return continuations

In [ ]:
# Cell 8 — Verifier (Step 4)
import re, signal
from math_verify import parse, verify

def extract_boxed(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    return matches[-1] if matches else ""

def verify_answer(pred: str, gold: str, timeout_sec: int = 5) -> int:
    def _handler(s, f): raise TimeoutError
    old = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(timeout_sec)
    try:
        pred_ans = extract_boxed(pred)
        if not pred_ans: return 0
        p = parse(pred_ans); g = parse("$" + str(gold) + "$")
        return int(bool(verify(g, p)))
    except Exception:
        return 0
    finally:
        signal.alarm(0); signal.signal(signal.SIGALRM, old)

def compute_rewards(continuations: list, answer: str) -> list:
    rewards = []
    for y in continuations:
        r = float(verify_answer(y, answer))
        if LAMBDA_LEN > 0:
            r -= LAMBDA_LEN * len(y.split())
        rewards.append(r)
    return rewards

In [ ]:
# Cell 9 — Group-Relative Advantage (Step 5)
import math

def compute_advantage(rewards: list) -> list:
    mu = sum(rewards) / len(rewards)
    variance = sum((r - mu) ** 2 for r in rewards) / len(rewards)
    sigma = math.sqrt(variance) + EPSILON
    return [(r - mu) / sigma for r in rewards]

def classify_group(rewards: list) -> str:
    binary = [1 if r > 0 else 0 for r in rewards]
    if all(b == 1 for b in binary): return "all_correct"
    if all(b == 0 for b in binary): return "all_wrong"
    return "mixed"

In [ ]:
# Cell 10 — Prefix-Feedback Loop (Step 6)
def sentence_split(text: str) -> list:
    parts = text.split(". ")
    sentences = []
    for s in parts:
        s = s.strip()
        if s:
            sentences.append(s if s.endswith(".") else s + ".")
    return sentences

def build_prefix_ladder(row: dict) -> list:
    full_cot = row.get("full_cot", "")
    sentences = sentence_split(full_cot)
    return [" ".join(sentences[:i+1]) for i in range(len(sentences))]

def get_next_prefix(current_prefix: str, ladder: list):
    cur_len = len(current_prefix)
    for candidate in ladder:
        if len(candidate) > cur_len:
            return candidate
    return None

In [ ]:
# Cell 11 — Main Data-Building Loop
import json
from tqdm import tqdm

SFT_L1_FILE  = f"{BASE_DIR}/data/sft_data_L1.jsonl"
SFT_L2_FILE  = f"{BASE_DIR}/data/sft_data_L2.jsonl"
DPO_FILE     = f"{BASE_DIR}/data/dpo_pairs.jsonl"
STATS_FILE   = f"{BASE_DIR}/data/build_stats.json"

# Resume: collect already-processed source IDs
processed_ids = set()
for fpath in [SFT_L1_FILE, SFT_L2_FILE]:
    if os.path.exists(fpath):
        with open(fpath) as f:
            for line in f:
                d = json.loads(line)
                processed_ids.add(d.get("source_id", ""))
print(f"Already processed: {len(processed_ids)} source IDs")

with open(PREFIX_FILE) as f:
    rows = [json.loads(l) for l in f]

stats = {"total": 0, "all_correct": 0, "all_wrong": 0, "mixed": 0, "feedback_triggered": 0}

f_l1  = open(SFT_L1_FILE,  "a", encoding="utf-8")
f_l2  = open(SFT_L2_FILE,  "a", encoding="utf-8")
f_dpo = open(DPO_FILE,     "a", encoding="utf-8")

try:
    for row in tqdm(rows, desc="Building dataset"):
        src_id = str(row.get("id", row["question"][:40]))
        if src_id in processed_ids:
            continue

        question = row["question"]
        prefix   = row["prefix"]
        answer   = str(row["answer"])
        stats["total"] += 1

        ladder = build_prefix_ladder(row)

        # Step 6: prefix-feedback loop
        for attempt in range(T_MAX):
            continuations = sample_k_continuations(question, prefix, K)
            rewards       = compute_rewards(continuations, answer)
            pass_rate     = sum(1 for r in rewards if r > 0) / K

            if pass_rate < TAU_LOW:
                next_p = get_next_prefix(prefix, ladder)
                if next_p:
                    prefix = next_p
                    stats["feedback_triggered"] += 1
                    continue
            break

        group_type = classify_group(rewards)
        stats[group_type] = stats.get(group_type, 0) + 1

        if group_type == "all_wrong":
            continue

        advantages = compute_advantage(rewards)

        mu_r     = sum(rewards) / len(rewards)
        sum_w_l1 = sum(1.0 for r in rewards if r >= mu_r) + EPSILON
        sum_w_l2 = sum(max(advantages[k], 0) for k in range(K)) + EPSILON

        for k, (y, r, a_k) in enumerate(zip(continuations, rewards, advantages)):
            full_seq = prefix + "\n" + y
            w_l1 = (1.0 / sum_w_l1) if r >= mu_r else 0.0
            w_l2 = min(max(a_k, 0), CLIP_C) / sum_w_l2

            base = {
                "source_id": src_id, "question": question,
                "prefix": prefix, "continuation": y,
                "full_sequence": full_seq, "answer": answer,
                "reward": r, "pass_rate": pass_rate,
            }
            if w_l1 > 0:
                f_l1.write(json.dumps({**base, "weight": w_l1}, ensure_ascii=False) + "\n")
            if w_l2 > 0:
                f_l2.write(json.dumps({**base, "weight": w_l2}, ensure_ascii=False) + "\n")

        if group_type == "mixed":
            best_k  = max(range(K), key=lambda k: rewards[k])
            worst_k = min(range(K), key=lambda k: rewards[k])
            if rewards[best_k] > rewards[worst_k]:
                f_dpo.write(json.dumps({
                    "source_id": src_id, "question": question, "prefix": prefix,
                    "chosen": continuations[best_k], "rejected": continuations[worst_k],
                    "reward_chosen": rewards[best_k], "reward_rejected": rewards[worst_k],
                }, ensure_ascii=False) + "\n")
finally:
    f_l1.close(); f_l2.close(); f_dpo.close()

with open(STATS_FILE, "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

In [ ]:
# Cell 12 — Sanity Check & Stats
import json

for label, fpath in [("L1", SFT_L1_FILE), ("L2", SFT_L2_FILE), ("DPO", DPO_FILE)]:
    with open(fpath) as f:
        lines = f.readlines()
    print(f"{label}: {len(lines)} samples in {fpath}")

with open(STATS_FILE) as f:
    print(json.load(f))